In [ ]:
import os, json, random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    set_seed,
)
try:
    from transformers.optimization import get_linear_schedule_with_warmup
except Exception:
    get_linear_schedule_with_warmup = None

from huggingface_hub import snapshot_download

BASE_MODEL = "nghuyong/ernie-3.0-base-zh"
LOCAL_BASE_DIR = "ernie-base-local"
OUTPUT_DIR = "ernie-domain-clf"
DATA_PATH = "wmt2425_en_zh_CN.jsonl"

TEXT_FIELD = "target"
LABEL_FIELD = "domain"
LABELS = ["news", "social", "speech", "literary"]

MAX_LEN = 256
SEED = 42
BATCH_TRAIN = 16
BATCH_EVAL = 32
LR = 2e-5
EPOCHS = 5
WARMUP_RATIO = 0.1

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

local_model_path = snapshot_download(repo_id=BASE_MODEL, local_dir=LOCAL_BASE_DIR)
print(f"Model downloaded to: {local_model_path}")

with open(DATA_PATH, encoding="utf-8") as f:
    raw = [json.loads(line) for line in f]

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

random.shuffle(raw)
n = len(raw)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_raw = raw[:n_train]
val_raw = raw[n_train:n_train + n_val]
test_raw = raw[n_train + n_val:]

class ZhDomainDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, text_field, label_field, label2id):
        self.rows = rows
        self.tok = tokenizer
        self.max_len = max_len
        self.text_field = text_field
        self.label_field = label_field
        self.label2id = label2id

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        ex = self.rows[i]
        enc = self.tok(
            str(ex[self.text_field]),
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.label2id[ex[self.label_field]])
        return enc

tokenizer = AutoTokenizer.from_pretrained(local_model_path, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    local_model_path,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
).to(device)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds = ZhDomainDataset(train_raw, tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)
val_ds = ZhDomainDataset(val_raw, tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)
test_ds = ZhDomainDataset(test_raw, tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)

train_loader = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=BATCH_EVAL, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=BATCH_EVAL, shuffle=False, collate_fn=collator)

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * max(1, EPOCHS)
warmup_steps = int(WARMUP_RATIO * total_steps)

if get_linear_schedule_with_warmup is not None and total_steps > 0:
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
else:
    scheduler = None

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def evaluate(dataloader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
        logits = outputs.logits
        all_preds.append(logits.argmax(-1).detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "y_true": y_true,
        "y_pred": y_pred,
    }

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for step, batch in enumerate(progress, 1):
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        avg_loss = running_loss / step
        progress.set_postfix(loss=f"{avg_loss:.4f}")

    val_metrics = evaluate(val_loader)
    print(f"Epoch {epoch} | Val acc {val_metrics['accuracy']:.2f} | Val F1 {val_metrics['f1_macro']:.2f}")

test_metrics = evaluate(test_loader)
print("Test accuracy:", test_metrics["accuracy"])
print("Test macro-F1:", test_metrics["f1_macro"])

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ds = load_dataset("haoranxu/ALMA-Human-Parallel", "zh-en")['train']

MODEL_DIR = "ernie-domain-clf"
OUT_DIR   = "./domain_splits_train"
BATCH_SIZE = 32

CLASSIFY_MAX_LEN = 256

SYSTEM_PROMPT = (
    "You are a translation assistant. Translate Chinese to English and output only the translation."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()
id2label = model.config.id2label

zh_lines = [ln['translation']['zh'].strip() for ln in ds]
en_lines = [ln['translation']['en'].strip() for ln in ds]


all_preds = [None] * len(zh_lines)
for i in range(0, len(zh_lines), BATCH_SIZE):
    batch_texts = zh_lines[i:i+BATCH_SIZE]
    enc = tok(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=CLASSIFY_MAX_LEN
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1).tolist()


    for k, p in enumerate(preds):
        all_preds[i + k] = id2label[p]

assert all(x is not None for x in all_preds)

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
buckets = {}

for idx, (zh, en, domain) in enumerate(zip(zh_lines, en_lines, all_preds)):
    rec = {
        "idx": idx,
        "domain": domain,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": zh},
            {"role": "assistant", "content": en},
        ],
    }
    buckets.setdefault(domain, []).append(rec)

for domain, recs in buckets.items():
    recs.sort(key=lambda r: r["idx"])
    out_path = out_dir / f"{domain}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(buckets)} domain files to {out_dir.resolve()}")
for d, recs in buckets.items():
    print(f" - {d}: {len(recs)}")